# Pipeline de execução experimental — Treinamento, Teste e Persistência de Resultados

Usado para a execução de cada experimento no ambiente do Google Colab.

<a href="https://colab.research.google.com/github/amartinsmg/classification-of-medical-images-using-cnn/blob/main/notebooks/experiment_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Preparação do ambiente


### Verificação do ambiente de execução

- Validação da GPU
- Documentação do ambiente de execução

In [1]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

!nvidia-smi

2.20.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Sat May  9 19:38:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                       

### Clonagem do respositório atualizado do GitHub

- Clona o repositório via `git clone`
- Transfere o ambiente de execução para a pasta raiz do repositório

In [ ]:
%cd /content
!git clone https://github.com/amartinsmg/classification-of-medical-images-using-cnn.git
%cd /content/classification-of-medical-images-using-cnn

### Montagem do Google Drive

- Monta o Google Drive permitindo que os arquivos presentes nele sejam lidos e escritos
- Define o diretório base de onde serão carregados os dados de treino, validação e teste, e onde serão salvos o modelo e os resultados do experimento

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
BASE_PATH = "/content/drive/MyDrive/classification-of-medical-images-using-cnn/"

## Configuração do experimento

Define os hiperparâmetros e identificadores do experimento a ser executado.

- `EXPERIMENT_NAME`: identificador único do experimento, usado como nome da subpasta em `results/` e `models/`
- `BASE_MODEL`: arquitetura do modelo base. Valores: `"resnet"`, `"densenet"`, `"efficientnet"`
- `NORMALIZATION`: estratégia de normalização. Valores: `"rescaling"` (pixels/255) ou `"preprocess-input"` (normalização específica da arquitetura)
- `DATA_AUG`: aplica flip horizontal, rotação e zoom aleatórios durante o treino
- `CLASS_WEIGHT`: aplica pesos inversamente proporcionais à frequência de cada classe para compensar o desbalanceamento do dataset
- `LEARNING_RATE`: taxa de aprendizado do otimizador Adam
- `EPOCHS`: número de épocas de treinamento
- `THRESHOLD`: limiar de decisão para classificação binária na avaliação
- `seeds`: lista de seeds para as 3 runs independentes, garantindo reprodutibilidade estatística

In [4]:
EXPERIMENT_NAME = "efficientnet-final-2"
BASE_MODEL = "efficientnet"
NORMALIZATION = "preprocess-input"
DATA_AUG = True
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_WEIGHT = True
LEARNING_RATE = 0.001
EPOCHS = 10
THRESHOLD = 0.5
seeds = [42, 123, 999]

## Configuração dos *paths*

Configuração dos caminhos para os diretórios onde serão salvos os resultados e os modelos

In [5]:
RESULTS_RELATIVE_PATH = "results/" + EXPERIMENT_NAME
RESULTS_LOCAL_PATH = BASE_PATH + "/" + RESULTS_RELATIVE_PATH
MODELS_RELATIVE_PATH = "models/" + EXPERIMENT_NAME
MODELS_LOCAL_PATH = BASE_PATH + "/" + MODELS_RELATIVE_PATH

## Obtenção da engine do banco de dados

Foi usado um banco de dados sqlite hospedado também no Google Drive

In [6]:
from src.db import insert_run, get_engine

DB_LOCAL_PATH = f"{BASE_PATH}/experiments.db"
DB_PATH = f"sqlite:///{DB_LOCAL_PATH}"

engine = get_engine(DB_PATH)

# Execução do experimento

### Realização inicial das três rodadas do experimento

- Treinamento do modelo com seeds diferentes
- Teste do modelo treinado, salvando apenas as predições do modelo

In [ ]:
from src.train import train_pipeline
from src.test import test_pipeline

configs = []

for i in range(3):
    run_id = i + 1

    config = train_pipeline(
        base_dir=BASE_PATH,
        experiment_name=EXPERIMENT_NAME,
        run_id=run_id,
        normalization=NORMALIZATION,
        base_model=BASE_MODEL,
        data_augmentation=DATA_AUG,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_weights=CLASS_WEIGHT,
        learning_rate=LEARNING_RATE,
        epochs=EPOCHS,
        seed=seeds[i],
        versioning_models=True,
    )
    configs.append(config)

    _ = test_pipeline(
        base_dir=BASE_PATH,
        experiment_name=EXPERIMENT_NAME,
        run_id=run_id,
        normalization=NORMALIZATION,
        base_model=BASE_MODEL,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        versioning_models=True,
        save_predictions=True,
    )

Found 5216 files belonging to 2 classes.
Found 16 files belonging to 2 classes.
Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 50s 124ms/step - AUC: 0.9713 - accuracy: 0.9055 - loss: 0.2143 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 0.1166
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - AUC: 0.9880 - accuracy: 0.9425 - loss: 0.1368 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 0.0592
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 38ms/step - AUC: 0.9928 - accuracy: 0.9578 - loss: 0.1031 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 0.0611
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - AUC: 0.9930 - accuracy: 0.9594 - loss: 0.1026 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 0.0475
Epoch 5/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 38ms/step - AUC: 0.9934 - accuracy: 0.9613 - loss: 0.0966 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 0.0530
Epoch 6/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - AUC: 0.9949 - accuracy: 0.9630 - loss: 0.0850 - val_AUC: 1

### Cálculo do limiar ideal
Usa as predições salvas para calcular o threshold ideal

In [8]:
from src.utils import optimal_threshold

THRESHOLD = optimal_threshold(RESULTS_LOCAL_PATH)

print(f"Optimal threshold: {THRESHOLD}")

Optimal threshold: 0.9202832


### Realização das três rodadas finais do experimento

- Teste do modelo com o threshold ideal e salvamento das métricas no Google Drive
- Salvamento das configurações e das métricas no banco de dados

In [ ]:
for i in range(3):
    run_id = i + 1

    metrics = test_pipeline(
      base_dir=BASE_PATH,
      experiment_name=EXPERIMENT_NAME,
      run_id=run_id,
      normalization=NORMALIZATION,
      base_model=BASE_MODEL,
      image_size=IMAGE_SIZE,
      batch_size=BATCH_SIZE,
      threshold=THRESHOLD,
      versioning_models=True,
    )

    insert_run(
        engine=engine,
        experiment=EXPERIMENT_NAME,
        run_name=f"run-{run_id}",
        config=configs[i],
        metrics=metrics,
    )

Found 624 files belonging to 2 classes.
20/20 ━━━━━━━━━━━━━━━━━━━━ 15s 455ms/step

 Experiment Summary (efficientnet-final-2: run 1)
+----------------------+------------+-------------+----------+------------+---------------+-----------+
|   Decision Threshold |   Accuracy |   Precision |   Recall |   F1-Score |   Specificity |   AUC ROC |
+======================+============+=============+==========+============+===============+===========+
|             0.920283 |   0.899038 |    0.907731 | 0.933333 |   0.920354 |       0.84188 |  0.962404 |
+----------------------+------------+-------------+----------+------------+---------------+-----------+
Test completed. Metrics saved.
Run 'run-1' of experiment 'efficientnet-final-2' inserted.
Found 624 files belonging to 2 classes.
20/20 ━━━━━━━━━━━━━━━━━━━━ 16s 453ms/step

 Experiment Summary (efficientnet-final-2: run 2)
+----------------------+------------+-------------+----------+------------+---------------+-----------+
|   Decision Thresho

## Envio dos resultados para o DagsHub via DVC

- Instala DagsHub para o controle de versão dos arquivos e para o upload dos resultados
- Loga no DagsHub com Secrets do Google Colab
- Faz upload da pasta com os modelos treinados no último experimento

In [ ]:
%pip install -q dvc dagshub

import dagshub
from google.colab import userdata

dagshub.auth.add_app_token(token=userdata.get("DAGSHUB_TOKEN"))

DAGSHUB_REPO = "amartinsmg/classification-of-medical-images-using-cnn"

dagshub.upload_files(
    DAGSHUB_REPO,
    local_path=MODELS_LOCAL_PATH,
    remote_path=MODELS_RELATIVE_PATH,
)

Upload da pasta com os resultados do último experimento feito

In [ ]:
dagshub.upload_files(
    DAGSHUB_REPO,
    local_path=RESULTS_LOCAL_PATH,
    remote_path=RESULTS_RELATIVE_PATH,
)

Copia o banco de dados do Drive para o DagsHub

In [ ]:
dagshub.upload_files(
    DAGSHUB_REPO,
    local_path=DB_LOCAL_PATH,
    remote_path="experiments.db",
    force=True
)

## Exemplo da estrura de arquivos gerados por este pipeline

- Estes dados foram salvos no Google Drive durante a sessão no Colab, garantindo sua persistencia mesmo diante de uma queda na sessão.
- Ao final do pipeline, foram enviados ao DagsHub, permitindo seu versionamento e transmissão via protocolo DVC (Data Version Control).

```text
├── experiments.db                   ← registro da run no banco de dados
├── models
│   └── experiment-name
│       └── run0X
│           ├── model.keras          ← modelo serializado
│           └── model.weights.h5     ← pesos isolados (redundante com .keras, mantido como backup)
└── results
    └── experiment-name
        └── run0X
            ├── config.json          ← hiperparâmetros da run
            ├── history.csv          ← histórico de treinamento
            ├── predictions.csv      ← predições brutas, usadas para calcular o threshold ideal
            └── metrics.json         ← métricas calculadas com o threshold ideal
```